In [ ]:
!pip install kaggle -q

from google.colab import files
uploaded = files.upload()

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plant_data --unzip

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [00:25<00:00, 113MB/s] 



In [ ]:
import os

for root, dirs, files_ in os.walk("/content/plant_data"):
    level = root.replace("/content/plant_data", "").count(os.sep)
    if level < 3:
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")

plant_data/
  test/
    test/
  New Plant Diseases Dataset(Augmented)/
    New Plant Diseases Dataset(Augmented)/
  new plant diseases dataset(augmented)/
    New Plant Diseases Dataset(Augmented)/


In [ ]:
import tensorflow as tf

IMG_SIZE = 160
BATCH_SIZE = 32

train_dir = "/content/plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
val_dir = "/content/plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(num_classes)
print(class_names[:5])

Found 70295 files belonging to 38 classes.
Found 17572 files belonging to 38 classes.
38
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(train_ds, validation_data=val_ds, epochs=8)

Epoch 1/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 112s 47ms/step - accuracy: 0.8468 - loss: 0.5113 - val_accuracy: 0.9155 - val_loss: 0.2662
Epoch 2/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 135s 46ms/step - accuracy: 0.9091 - loss: 0.2798 - val_accuracy: 0.9144 - val_loss: 0.2584
Epoch 3/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 104s 48ms/step - accuracy: 0.9162 - loss: 0.2532 - val_accuracy: 0.9156 - val_loss: 0.2563
Epoch 4/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 138s 46ms/step - accuracy: 0.9187 - loss: 0.2470 - val_accuracy: 0.9249 - val_loss: 0.2262
Epoch 5/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 145s 47ms/step - accuracy: 0.9224 - loss: 0.2421 - val_accuracy: 0.9155 - val_loss: 0.2572
Epoch 6/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 101s 46ms/step - accuracy: 0.9250 - loss: 0.2363 - val_accuracy: 0.9244 - val_loss: 0.2325
Epoch 7/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 101s 46ms/step - accuracy: 0.9256 - loss: 0.2299 - val_accuracy: 0.9220 - val_loss: 0.2439
Epoch 8/8
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 103s 47ms/step - accuracy: 0.9273 -

In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 177s 75ms/step - accuracy: 0.8467 - loss: 0.6058 - val_accuracy: 0.9298 - val_loss: 0.2311
Epoch 2/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 162s 74ms/step - accuracy: 0.9229 - loss: 0.2366 - val_accuracy: 0.9464 - val_loss: 0.1709
Epoch 3/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 199s 73ms/step - accuracy: 0.9421 - loss: 0.1740 - val_accuracy: 0.9517 - val_loss: 0.1489
Epoch 4/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 159s 73ms/step - accuracy: 0.9550 - loss: 0.1319 - val_accuracy: 0.9597 - val_loss: 0.1265
Epoch 5/5
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 204s 73ms/step - accuracy: 0.9624 - loss: 0.1100 - val_accuracy: 0.9639 - val_loss: 0.1114


In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Final Validation Accuracy: {val_acc*100:.2f}%")

550/550 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - accuracy: 0.9639 - loss: 0.1114
Final Validation Accuracy: 96.39%


In [ ]:
model.save("plant_disease_classifier.keras")

import json
with open("plant_class_names.json", "w") as f:
    json.dump(class_names, f)

from google.colab import files
files.download("plant_disease_classifier.keras")
files.download("plant_class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>